In [1]:
gcs_connector_path = '../../config/gcs-connector-hadoop3-latest.jar'
bigquery_connector_path = '../../config/spark-bigquery-with-dependencies_2.12-0.35.0.jar'

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
        .master("local[*]") \
        .appName('Transform Stage (Kaggle)') \
        .config("spark.jars", f"{gcs_connector_path},{bigquery_connector_path}") \
        .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
        .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
        .getOrCreate()

26/04/04 12:50:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [19]:
BUCKET_NAME="mnl_accident_pipeline_bucket"
CLEANED_FOLDER_NAME="cleaned_v2"

fname = f"gs://{BUCKET_NAME}/{CLEANED_FOLDER_NAME}/date=2026-04-03/"

In [20]:
df = spark.read.parquet(fname)

In [22]:
df.show(truncate=False)

+-----+-----------+---------------------------------------+----------------+----------------+-------------+---------+------------------------------+-------------+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------+
|time |city       |location                               |latitude        |longitude       |high_accuracy|direction|type                          |lanes_blocked|involved          |tweet                                                                                                                                                                               |source                                         |
+-----+-----------+---------------------------------------+----------------+----------------+-------------+---------+------------------------------+-------------+----------------

In [4]:
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from google.cloud import storage

import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DateType

from utils.LocationFunctions import load_locations_df, get_locations_from_bq, get_missing_locations, get_batch_geocode, update_locations_bq
from utils.Common import gcs_file_read, gcs_upload_parquet, partial_parse_raw_data

In [5]:
load_dotenv()
RawSchema = StructType([
    StructField('content', StringType(), True),
    StructField('tweetlinkid', StringType(), True),
    StructField('created_at', DateType(), True),
])
def get_current_raw_filename(scrape_folder):
    PHT = ZoneInfo("Asia/Manila")
    now = datetime.now(PHT)
    yesterday = now - timedelta(days=1)

    year = yesterday.strftime("%Y")
    month = yesterday.strftime("%m")
    day = yesterday.strftime("%d")

    filename = f"{scrape_folder}/{year}/{month}/scrape_data_{year}{month}{day}.csv"
    return filename

In [6]:
project_id = os.getenv("PROJECT_ID")
dataset = os.getenv("DATASET")
locations_table_id = f"{project_id}:{dataset}.locations"
staging_locations_table_id = f"{project_id}:{dataset}.locations_staging"
bucket_name = os.getenv('BUCKET_NAME')
raw_folder = os.getenv('RAW_FOLDER_NAME')
clean_folder = os.getenv('CLEANED_FOLDER_NAME')

scrape_folder = f"{raw_folder}/scrape"

In [7]:
gcs_client = storage.Client()
bucket = gcs_client.bucket(bucket_name)

df_locations = load_locations_df(spark, locations_table_id)
raw_filename = get_current_raw_filename(scrape_folder)
is_file_exists = storage.Blob(bucket=bucket, name=raw_filename).exists()

In [8]:
is_file_exists

True

In [9]:
df_raw = gcs_file_read(spark, bucket_name, raw_filename, RawSchema)

In [10]:
df_partial_parsed = partial_parse_raw_data(df_raw)

In [11]:
df_partial_parsed.show(5, truncate=False)

+----------+-----+---------------------------------------+---------+------------------------------+-------------+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------+
|date      |time |location                               |direction|type                          |lanes_blocked|involved          |tweet                                                                                                                                                                               |source                                         |
+----------+-----+---------------------------------------+---------+------------------------------+-------------+------------------+------------------------------------------------------------------------------------------------------------------------------------------------

get_time(): Raw input MMDA ALERT: ROAD CRASH INCIDENT AT COMMONWEALTH SANDIGAN TUNNEL WB INVOLVING A MOTORCYCLE TRUCK AS OF 7:06 AM ONE LANE OCCUPIED MMDA ENFORCERS ARE ON SITE MANAGING TRAFFIC #MMDA
get_time(): Parsed output 07:06
get_lanes_blocked(): Raw input MMDA ALERT: ROAD CRASH INCIDENT AT COMMONWEALTH SANDIGAN TUNNEL WB INVOLVING A MOTORCYCLE TRUCK AS OF 7:06 AM. ONE LANE OCCUPIED. MMDA ENFORCERS ARE ON SITE MANAGING TRAFFIC. #MMDA
get_lanes_blocked(): RegEx Match ONE LANE
get_lanes_blocked(): Cleaned output 1
get_inc_type(): Raw input MMDA ALERT: ROAD CRASH INCIDENT AT COMMONWEALTH SANDIGAN TUNNEL WB INVOLVING A MOTORCYCLE TRUCK AS OF 7:06 AM. ONE LANE OCCUPIED. MMDA ENFORCERS ARE ON SITE MANAGING TRAFFIC. #MMDA
get_inc_type(): RegEx Match MMDA ALERT: ROAD CRASH INCIDENT AT 
get_inc_type(): Cleaned output ROAD CRASH INCIDENT
get_location(): RegEx Match  AT COMMONWEALTH SANDIGAN TUNNEL WB INVOLVING A MOTORCYCLE TRUCK AS OF 7:06 AM
get_location(): Cleaned Location COMMONWEALTH S

In [12]:
df_full_parsed = get_locations_from_bq(df_locations, df_partial_parsed)

In [13]:
missing_locations = get_missing_locations(df_full_parsed)

get_time(): Raw input MMDA ALERT: ROAD CRASH INCIDENT AT COMMONWEALTH SANDIGAN TUNNEL WB INVOLVING A MOTORCYCLE TRUCK AS OF 7:06 AM ONE LANE OCCUPIED MMDA ENFORCERS ARE ON SITE MANAGING TRAFFIC #MMDA
get_time(): Parsed output 07:06
get_lanes_blocked(): Raw input MMDA ALERT: ROAD CRASH INCIDENT AT COMMONWEALTH SANDIGAN TUNNEL WB INVOLVING A MOTORCYCLE TRUCK AS OF 7:06 AM. ONE LANE OCCUPIED. MMDA ENFORCERS ARE ON SITE MANAGING TRAFFIC. #MMDA
get_lanes_blocked(): RegEx Match ONE LANE
get_lanes_blocked(): Cleaned output 1
get_inc_type(): Raw input MMDA ALERT: ROAD CRASH INCIDENT AT COMMONWEALTH SANDIGAN TUNNEL WB INVOLVING A MOTORCYCLE TRUCK AS OF 7:06 AM. ONE LANE OCCUPIED. MMDA ENFORCERS ARE ON SITE MANAGING TRAFFIC. #MMDA
get_inc_type(): RegEx Match MMDA ALERT: ROAD CRASH INCIDENT AT 
get_inc_type(): Cleaned output ROAD CRASH INCIDENT
get_location(): RegEx Match  AT COMMONWEALTH SANDIGAN TUNNEL WB INVOLVING A MOTORCYCLE TRUCK AS OF 7:06 AM
get_location(): Cleaned Location COMMONWEALTH S

In [14]:
missing_locations

['COMMONWEALTH AFTER SANDIGAN U-TURN SLOT', 'C5 KALAYAAN AVE. (BAHAY BULILIT)']

In [17]:
resolved_locations_df = get_batch_geocode(spark, missing_locations)

In [18]:
resolved_locations_df.show(truncate=False)

+-----------+---------------------------------------+----------------+----------------+-------------+
|City       |Location                               |Latitude        |Longitude       |High_Accuracy|
+-----------+---------------------------------------+----------------+----------------+-------------+
|TAYTAY     |COMMONWEALTH AFTER SANDIGAN U-TURN SLOT|14.5670364000052|121.141459750006|1.0          |
|QUEZON CITY|C5 KALAYAAN AVE. (BAHAY BULILIT)       |14.6487588364638|121.051144150926|0.0          |
+-----------+---------------------------------------+----------------+----------------+-------------+

